In [1]:
import os
import shutil
from google.colab import drive

# Mount your Google Drive
print("Mounting Google Drive")
drive.mount('/content/drive')

# Install Libraries
print("Installing required libraries")
!pip install -q ftfy regex tqdm
!pip install -q git+https://github.com/openai/CLIP.git


Mounting Google Drive
Mounted at /content/drive
Installing required libraries
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import glob

# Define Project Paths
PROJECT_BASE_PATH = "/content/drive/My Drive/NSTV_FDL_Project"
DATASET_DIR = os.path.join(PROJECT_BASE_PATH, "Dataset")
INPUT_DIR = os.path.join(PROJECT_BASE_PATH, "Input")
MODEL_SAVE_DIR = os.path.join(PROJECT_BASE_PATH, "Trained_Models")
OUTPUT_IMAGE_DIR = os.path.join(PROJECT_BASE_PATH, "Output")
LOGS_DIR = os.path.join(PROJECT_BASE_PATH, "Logs")

# Create Project Directories
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
os.makedirs(OUTPUT_IMAGE_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

# Define Content Dataset Path
CONTENT_DATASET_PATH = "/content/coco_dataset/"
os.makedirs(CONTENT_DATASET_PATH, exist_ok=True)


# DYNAMIC PROMPTS
PROMPT_MAP = {
    "cyberpunk_xl": "A cinematic photo in the cyberpunk style, masterpiece, neon lights, high contrast, gritty realism, futuristic",
    "cyberpunk_mini": "A cinematic photo in the cyberpunk style, masterpiece",
    "war": "cinematic film still from a war drama, gritty realism, desaturated colors, high contrast, intense atmosphere, film grain, masterpiece, sharp focus",
    "imax": "A cinematic photo in the Hollywood IMAX style, high detail, epic scale, sharp focus"
}

# Automatically Find All Training Jobs
all_dataset_zips = glob.glob(os.path.join(DATASET_DIR, "*.zip"))
training_jobs = []

for zip_path in all_dataset_zips:
    style_name = os.path.basename(zip_path).split('.')[0]
    text_prompt = PROMPT_MAP.get(style_name.lower())

    if text_prompt:
        job = {
            "style_name": style_name,
            "zip_path": zip_path,
            "text_prompt": text_prompt,
            "model_output_path": os.path.join(MODEL_SAVE_DIR, f"{style_name}_net.pth"),
            "style_images_path": f"/content/{style_name}_images/",
            "inference_output_path": os.path.join(OUTPUT_IMAGE_DIR, f"inference_{style_name}.jpg"),
            "log_file_path": os.path.join(LOGS_DIR, f"{style_name}_training_log.txt"),
            "plot_file_path": os.path.join(LOGS_DIR, f"{style_name}_loss_plot.png")
        }
        training_jobs.append(job)
        os.makedirs(job["style_images_path"], exist_ok=True)
    else:
        print(f"Warning: Found '{zip_path}' but no matching prompt in PROMPT_MAP. Skipping.")

print(f"Complete: Found {len(training_jobs)} training jobs.")
print("Jobs to run:")
for job in training_jobs:
    print(f"- {job['style_name']} (Log will be saved to {job['log_file_path']})")

Complete: Found 4 training jobs.
Jobs to run:
- Cyberpunk_Mini (Log will be saved to /content/drive/My Drive/NSTV_FDL_Project/Logs/Cyberpunk_Mini_training_log.txt)
- War (Log will be saved to /content/drive/My Drive/NSTV_FDL_Project/Logs/War_training_log.txt)
- IMAX (Log will be saved to /content/drive/My Drive/NSTV_FDL_Project/Logs/IMAX_training_log.txt)
- Cyberpunk_XL (Log will be saved to /content/drive/My Drive/NSTV_FDL_Project/Logs/Cyberpunk_XL_training_log.txt)


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvLayer(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride):
        super(ConvLayer, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding=kernel_size//2, padding_mode='reflect')

    def forward(self, x):
        return self.conv(x)

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = ConvLayer(channels, channels, kernel_size=3, stride=1)
        self.in1 = nn.InstanceNorm2d(channels, affine=True)
        self.conv2 = ConvLayer(channels, channels, kernel_size=3, stride=1)
        self.in2 = nn.InstanceNorm2d(channels, affine=True)
        self.relu = nn.ReLU()

    def forward(self, x):
        residual = x
        out = self.relu(self.in1(self.conv1(x)))
        out = self.in2(self.conv2(out))
        out = out + residual
        return out

class UpsampleConvLayer(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, upsample=None):
        super(UpsampleConvLayer, self).__init__()
        self.upsample = upsample
        self.conv = ConvLayer(in_channels, out_channels, kernel_size, stride)

    def forward(self, x):
        if self.upsample:
            x = F.interpolate(x, mode='nearest', scale_factor=self.upsample)
        return self.conv(x)

class TransformerNet(nn.Module):
    def __init__(self):
        super(TransformerNet, self).__init__()
        # Encoder
        self.conv1 = ConvLayer(3, 32, kernel_size=9, stride=1)
        self.in1 = nn.InstanceNorm2d(32, affine=True)
        self.conv2 = ConvLayer(32, 64, kernel_size=3, stride=2)
        self.in2 = nn.InstanceNorm2d(64, affine=True)
        self.conv3 = ConvLayer(64, 128, kernel_size=3, stride=2)
        self.in3 = nn.InstanceNorm2d(128, affine=True)
        # Residual Blocks
        self.res1 = ResidualBlock(128)
        self.res2 = ResidualBlock(128)
        self.res3 = ResidualBlock(128)
        self.res4 = ResidualBlock(128)
        self.res5 = ResidualBlock(128)
        # Decoder
        self.deconv1 = UpsampleConvLayer(128, 64, kernel_size=3, stride=1, upsample=2)
        self.in4 = nn.InstanceNorm2d(64, affine=True)
        self.deconv2 = UpsampleConvLayer(64, 32, kernel_size=3, stride=1, upsample=2)
        self.in5 = nn.InstanceNorm2d(32, affine=True)
        self.deconv3 = ConvLayer(32, 3, kernel_size=9, stride=1)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.relu(self.in1(self.conv1(x)))
        out = self.relu(self.in2(self.conv2(out)))
        out = self.relu(self.in3(self.conv3(out)))
        out = self.res1(out)
        out = self.res2(out)
        out = self.res3(out)
        out = self.res4(out)
        out = self.res5(out)
        out = self.relu(self.in4(self.deconv1(out)))
        out = self.relu(self.in5(self.deconv2(out)))
        out = self.deconv3(out)
        return out


In [4]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from PIL import Image
import torch.optim as optim
import clip
from tqdm.notebook import tqdm
import random

# Training Parameters
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 5
BATCH_SIZE = 4
IMAGE_SIZE = 256
CONTENT_WEIGHT = 1e5
STYLE_WEIGHT = 1e10
CLIP_WEIGHT = 1e3
LEARNING_RATE = 1e-3

# Data Loading Helper
def get_image_paths(directory):
    paths = []
    for root, _, files in os.walk(directory):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                paths.append(os.path.join(root, file))
    return paths

# VGG & CLIP Models
print("Loading VGG and CLIP models.")
vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT).features.to(DEVICE).eval()
for param in vgg.parameters(): param.requires_grad = False

clip_model, clip_preprocess = clip.load("ViT-B/32", device=DEVICE)
for param in clip_model.parameters(): param.requires_grad = False
print("VGG and CLIP models loaded.")


# Loss Functions
def get_vgg_features(image, model, layers=None):
    if layers is None:
        layers = {'3': 'relu1_2', '8': 'relu2_2', '15': 'relu3_3', '22': 'relu4_3'}
    features = {}
    x = image
    # VGG preprocessing
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(DEVICE)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(DEVICE)
    x = (x / 255.0 - mean) / std

    for name, layer in model._modules.items():
        x = layer(x)
        if name in layers:
            features[layers[name]] = x
    return features

def gram_matrix(y):
    (b, c, h, w) = y.size()
    features = y.view(b, c, w * h)
    features_t = features.transpose(1, 2)
    gram = features.bmm(features_t) / (c * h * w)
    return gram

# CLIP Preprocessing Function
def preprocess_for_clip(image):
    # CLIP's specific normalization
    clip_mean = torch.tensor([0.48145466, 0.4578275, 0.40821073]).to(DEVICE).view(1,3,1,1)
    clip_std = torch.tensor([0.26862954, 0.26130258, 0.27577711]).to(DEVICE).view(1,3,1,1)
    image = F.interpolate(image, 224, mode='bicubic', align_corners=False)
    image = (image / 255.0 - clip_mean) / clip_std
    return image


Loading VGG and CLIP models.
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:04<00:00, 116MB/s]
100%|███████████████████████████████████████| 338M/338M [00:04<00:00, 86.2MiB/s]


VGG and CLIP models loaded.


In [5]:
import os

# Download and Unzip the Content Dataset (MS-COCO)
if not os.path.exists(os.path.join(CONTENT_DATASET_PATH, "val2017")):
    print("Downloading content dataset.")
    !wget -q http://images.cocodataset.org/zips/val2017.zip -P /content/

    print("Unzipping content dataset.")
    !unzip -q -o /content/val2017.zip -d {CONTENT_DATASET_PATH}

    !rm /content/val2017.zip
    print("Content dataset ready.")
else:
    print("Content dataset (MS-COCO) already exists. Skipping download.")

# Prepare Content Loader
transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.mul(255))
])

content_dataset = datasets.ImageFolder(CONTENT_DATASET_PATH, transform)
content_loader = DataLoader(content_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"Content loader is ready with {len(content_dataset)} images.")

Unzipping content dataset.
Content dataset ready.
Content loader is ready with 5000 images.


In [6]:
from IPython.display import display
from PIL import Image
import time
import json
import matplotlib.pyplot as plt
import shutil

print("Starting main training and inference loop.")
print(f"Device: {DEVICE}")

# Path to your test inference image
inference_image_path = os.path.join(INPUT_DIR, "your_photo.jpg")

for job in training_jobs:
    style_name = job['style_name']
    model_path = job['model_output_path']
    log_file_path = job['log_file_path']
    plot_file_path = job['plot_file_path']

    print("\n" + "="*50)
    print(f"PROCESSING JOB: {style_name}")
    print(f"PROMPT: {job['text_prompt']}")
    print("="*50 + "\n")

    # Check if model and log already exist
    if os.path.exists(model_path) and os.path.exists(log_file_path):
        print(f"Found existing model: {model_path}")
        print(f"Found existing log: {log_file_path}")
        print("SKIPPING TRAINING")

        # Try to load and display info from the log
        try:
            with open(log_file_path, 'r') as f:
                log_data = json.load(f)
            print(f"Loaded log from: {log_data.get('timestamp_end_utc', 'N/A')}")
            print(f"Previous training time: {log_data.get('total_training_time_seconds', 'N/A')}s")

            # Display the existing loss plot if it exists
            if os.path.exists(plot_file_path):
                print("Displaying existing loss plot.")
                display(Image.open(plot_file_path))
            else:
                print("Note: Existing loss plot not found.")

        except Exception as e:
            print(f"Warning: Could not read log file: {e}")

    else:
        # Run Full Training
        print(f"Model or log not found. STARTING NEW TRAINING for {style_name}.")
        print(f"LOG FILE: {log_file_path}")

        # Log Setup
        log_data = {
            "job_name": style_name,
            "device": DEVICE,
            "timestamp_start_utc": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
            "hyperparameters": {
                "epochs": EPOCHS, "batch_size": BATCH_SIZE, "image_size": IMAGE_SIZE,
                "content_weight": CONTENT_WEIGHT, "style_weight": STYLE_WEIGHT,
                "clip_weight": CLIP_WEIGHT, "learning_rate": LEARNING_RATE
            },
            "paths": {
                "zip_path": job['zip_path'], "model_output_path": job['model_output_path'],
            },
            "prompt": job['text_prompt'],
            "epoch_logs": []
        }
        job_start_time = time.time()

        # 1. Create Dir and Unzip Style Dataset
        os.makedirs(job["style_images_path"], exist_ok=True) # <-- Create temp dir
        print(f"Unzipping {style_name} style images...")
        !unzip -q -o "{job['zip_path']}" -d "{job['style_images_path']}"

        # 2. Prepare Style Image & Losses
        style_image_paths = get_image_paths(job['style_images_path'])
        if not style_image_paths:
            print(f"Error: No images found in {job['style_images_path']}. Skipping job.")
            log_data["error"] = f"No images found in {job['style_images_path']}"
            with open(log_file_path, 'w') as f:
                json.dump(log_data, f, indent=4)
            # Clean up the empty dir
            shutil.rmtree(job['style_images_path'])
            continue # Skip to the next job

        style_transform = transforms.Compose([transforms.Resize(IMAGE_SIZE), transforms.CenterCrop(IMAGE_SIZE), transforms.ToTensor()])
        style_image = style_transform(Image.open(random.choice(style_image_paths))).unsqueeze(0).to(DEVICE)

        style_features = get_vgg_features(style_image.mul(255), vgg)
        style_grams = {layer: gram_matrix(style_features[layer]) for layer in style_features}
        text_tokens = clip.tokenize([job['text_prompt']]).to(DEVICE)
        text_features = clip_model.encode_text(text_tokens).detach()

        # 3. Training Setup
        transformer = TransformerNet().to(DEVICE)
        optimizer = optim.Adam(transformer.parameters(), lr=LEARNING_RATE)
        mse_loss = torch.nn.MSELoss()

        print(f"Starting training for '{style_name}'.")
        for e in range(EPOCHS):
            epoch_start_time = time.time()
            transformer.train()
            agg_content_loss, agg_style_loss, agg_clip_loss = 0., 0., 0.

            progress_bar = tqdm(enumerate(content_loader), total=len(content_loader), desc=f"Epoch {e+1}/{EPOCHS}")

            for batch_id, (x, _) in progress_bar:
                optimizer.zero_grad()
                y = transformer(x.to(DEVICE))

                features_y = get_vgg_features(y, vgg)
                features_x = get_vgg_features(x.to(DEVICE), vgg)
                content_loss = mse_loss(features_y['relu2_2'], features_x['relu2_2'])

                style_loss = 0.
                for layer in style_grams:
                    gram_y = gram_matrix(features_y[layer])
                    gram_s = style_grams[layer].expand_as(gram_y)
                    style_loss += mse_loss(gram_y, gram_s)

                clip_input = preprocess_for_clip(y)
                image_features = clip_model.encode_image(clip_input)
                clip_loss = 1 - torch.cosine_similarity(text_features, image_features).mean()

                total_loss = CONTENT_WEIGHT * content_loss + STYLE_WEIGHT * style_loss + CLIP_WEIGHT * clip_loss
                total_loss.backward()
                optimizer.step()

                agg_content_loss += content_loss.item()
                agg_style_loss += style_loss.item()
                agg_clip_loss += clip_loss.item()

                progress_bar.set_postfix({
                    'content': f"{content_loss.item():.4f}",
                    'style': f"{style_loss.item():.4f}",
                    'clip': f"{clip_loss.item():.4f}"
                })

            epoch_duration = time.time() - epoch_start_time
            avg_content = agg_content_loss / len(content_loader)
            avg_style = agg_style_loss / len(content_loader)
            avg_clip = agg_clip_loss / len(content_loader)

            print(f"Epoch {e+1} Avg Losses: Content={avg_content:.4f}, Style={avg_style:.4f}, CLIP={avg_clip:.4f}, Duration: {epoch_duration:.2f}s")

            log_data["epoch_logs"].append({
                "epoch": e + 1,
                "avg_content_loss": round(avg_content, 6),
                "avg_style_loss": round(avg_style, 6),
                "avg_clip_loss": round(avg_clip, 6),
                "duration_seconds": round(epoch_duration, 2)
            })

        # 4. Save Model and Final Log
        total_training_time = time.time() - job_start_time
        transformer.eval().cpu()
        torch.save(transformer.state_dict(), model_path) # Use model_path

        print(f"\nTraining complete! Model saved to {model_path}")
        print(f"Total training time: {total_training_time:.2f}s")

        log_data["total_training_time_seconds"] = round(total_training_time, 2)
        log_data["timestamp_end_utc"] = time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())

        with open(log_file_path, 'w') as f:
            json.dump(log_data, f, indent=4)
        print(f"Log file saved to {log_file_path}")

        # 5. Generate and Save Loss Plot
        print(f"Generating loss plot...")
        try:
            epochs = [log['epoch'] for log in log_data['epoch_logs']]
            content_losses = [log['avg_content_loss'] for log in log_data['epoch_logs']]
            clip_losses = [log['avg_clip_loss'] for log in log_data['epoch_logs']]

            plt.figure(figsize=(10, 5))
            plt.plot(epochs, content_losses, label='Content Loss', marker='o')
            plt.plot(epochs, clip_losses, label='CLIP Loss', marker='o')
            plt.title(f'"{style_name}" Training Loss over {EPOCHS} Epochs')
            plt.xlabel('Epoch')
            plt.ylabel('Average Loss')
            plt.legend()
            plt.grid(True)
            plt.xticks(epochs)

            plt.savefig(plot_file_path) # Use plot_file_path
            print(f"Plot saved to {plot_file_path}")
            plt.show() # Display the plot in the notebook

        except Exception as e:
            print(f"Could not generate plot: {e}")

        # 7. Clean Up
        print(f"Cleaning up {job['style_images_path']} to save space")
        shutil.rmtree(job['style_images_path'])
        print(f"NEW TRAINING FOR {style_name} COMPLETE")

    # 6. Run Test Inference (This runs ALWAYS)
    print(f"Running test inference for {style_name}.")

    try:
        inference_model = TransformerNet()
        inference_model.load_state_dict(torch.load(model_path)) # Use model_path
        inference_model.to(DEVICE)
        inference_model.eval()

        content_image = Image.open(inference_image_path)
        content_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Lambda(lambda x: x.mul(255))
        ])
        content_image_tensor = content_transform(content_image).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            output = inference_model(content_image_tensor)

        output_image_data = output.cpu().squeeze(0).clamp(0, 255).numpy().transpose(1, 2, 0).astype("uint8")
        output_image_pil = Image.fromarray(output_image_data)
        output_image_pil.save(job['inference_output_path'])

        print(f"Inference complete! Image saved to {job['inference_output_path']}")
        display(output_image_pil)

    except Exception as e:
        print(f"Error during inference for {style_name}: {e}")
        print("Please check if the model file is valid or if the input image path is correct.")

    print(f"JOB {style_name} PROCESSED")

print("\n" + "="*50)
print("ALL JOBS PROCESSED.")
print("="*50)

Output hidden; open in https://colab.research.google.com to view.

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import os
import glob
import matplotlib.pyplot as plt

# TransformerNet Class Definition
# This is required to load your model files
class ConvLayer(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride):
        super(ConvLayer, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding=kernel_size//2, padding_mode='reflect')
    def forward(self, x): return self.conv(x)

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = ConvLayer(channels, channels, kernel_size=3, stride=1)
        self.in1 = nn.InstanceNorm2d(channels, affine=True)
        self.conv2 = ConvLayer(channels, channels, kernel_size=3, stride=1)
        self.in2 = nn.InstanceNorm2d(channels, affine=True)
        self.relu = nn.ReLU()
    def forward(self, x):
        residual = x
        out = self.relu(self.in1(self.conv1(x)))
        out = self.in2(self.conv2(out))
        out = out + residual
        return out

class UpsampleConvLayer(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, upsample=None):
        super(UpsampleConvLayer, self).__init__()
        self.upsample = upsample
        self.conv = ConvLayer(in_channels, out_channels, kernel_size, stride)
    def forward(self, x):
        if self.upsample:
            x = F.interpolate(x, mode='nearest', scale_factor=self.upsample)
        return self.conv(x)

class TransformerNet(nn.Module):
    def __init__(self):
        super(TransformerNet, self).__init__()
        self.conv1 = ConvLayer(3, 32, kernel_size=9, stride=1)
        self.in1 = nn.InstanceNorm2d(32, affine=True)
        self.conv2 = ConvLayer(32, 64, kernel_size=3, stride=2)
        self.in2 = nn.InstanceNorm2d(64, affine=True)
        self.conv3 = ConvLayer(64, 128, kernel_size=3, stride=2)
        self.in3 = nn.InstanceNorm2d(128, affine=True)
        self.res1 = ResidualBlock(128)
        self.res2 = ResidualBlock(128)
        self.res3 = ResidualBlock(128)
        self.res4 = ResidualBlock(128)
        self.res5 = ResidualBlock(128)
        self.deconv1 = UpsampleConvLayer(128, 64, kernel_size=3, stride=1, upsample=2)
        self.in4 = nn.InstanceNorm2d(64, affine=True)
        self.deconv2 = UpsampleConvLayer(64, 32, kernel_size=3, stride=1, upsample=2)
        self.in5 = nn.InstanceNorm2d(32, affine=True)
        self.deconv3 = ConvLayer(32, 3, kernel_size=9, stride=1)
        self.relu = nn.ReLU()
    def forward(self, x):
        out = self.relu(self.in1(self.conv1(x)))
        out = self.relu(self.in2(self.conv2(out)))
        out = self.relu(self.in3(self.conv3(out)))
        out = self.res1(out); out = self.res2(out); out = self.res3(out)
        out = self.res4(out); out = self.res5(out)
        out = self.relu(self.in4(self.deconv1(out)))
        out = self.relu(self.in5(self.deconv2(out)))
        out = self.deconv3(out)
        return out

# Configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PROJECT_BASE_PATH = "/content/drive/My Drive/NSTV_FDL_Project"
MODEL_SAVE_DIR = os.path.join(PROJECT_BASE_PATH, "Trained_Models")
INPUT_DIR = os.path.join(PROJECT_BASE_PATH, "Input")
OUTPUT_IMAGE_DIR = os.path.join(PROJECT_BASE_PATH, "Output_Images")
GENERALIZATION_GRID_DIR = os.path.join(OUTPUT_IMAGE_DIR, "Generalization_Grids")

# Create the new directory
os.makedirs(GENERALIZATION_GRID_DIR, exist_ok=True)


# Helper Function to run inference
def stylize_image(model, image_path):
    content_image = Image.open(image_path).convert("RGB")
    content_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.mul(255))
    ])
    content_tensor = content_transform(content_image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        output_tensor = model(content_tensor)

    output_data = output_tensor.cpu().squeeze(0).clamp(0, 255).numpy().transpose(1, 2, 0).astype("uint8")
    return Image.fromarray(output_data)

# 1. Load all trained models
print("Loading all trained models.")
model_paths = glob.glob(os.path.join(MODEL_SAVE_DIR, "*.pth"))
models = {}
for path in model_paths:
    style_name = os.path.basename(path).replace("_net.pth", "")
    model = TransformerNet()
    model.load_state_dict(torch.load(path))
    model.to(DEVICE)
    model.eval()
    models[style_name] = model
    print(f"Loaded: {style_name}")

# 2. Load all content images
content_paths = glob.glob(os.path.join(INPUT_DIR, "*.jpg")) + glob.glob(os.path.join(INPUT_DIR, "*.png"))
content_images = {}
for path in content_paths:
    image_name = os.path.basename(path)
    content_images[image_name] = Image.open(path).convert("RGB")
print(f"Loaded {len(content_images)} content images.")


# VISUALIZATION 1: Generalization Grid (Generate for ALL styles)
print("\nGenerating Generalization Grids for all styles.")
num_images = len(content_images)

for style_name, model_to_test in models.items():
    print(f"Generating grid for '{style_name}' style.")
    plt.figure(figsize=(15, num_images * 5))
    plt.suptitle(f"Generalization Grid for '{style_name}' Style", fontsize=20, y=1.02)

    for i, (image_name, pil_image) in enumerate(content_images.items()):
        styled_image = stylize_image(model_to_test, os.path.join(INPUT_DIR, image_name))

        # Plot Original
        ax = plt.subplot(num_images, 2, i*2 + 1)
        ax.imshow(pil_image)
        ax.set_title(f"Original: {image_name}")
        ax.axis("off")

        # Plot Styled
        ax = plt.subplot(num_images, 2, i*2 + 2)
        ax.imshow(styled_image)
        ax.set_title(f"Styled: {style_name}")
        ax.axis("off")

    plt.tight_layout()
    # Save plot to the new directory
    plot_path = os.path.join(GENERALIZATION_GRID_DIR, f"generalization_grid_{style_name}.jpg")
    plt.savefig(plot_path, bbox_inches='tight')
    plt.close() # Close the figure to free up memory
    print(f"Generalization grid saved to {plot_path}")


# VISUALIZATION 2: Style Comparison Grid
# Shows all styles on one content image
print("\nGenerating Style Comparison Grid.")
IMAGE_TO_TEST = "your_photo.jpg"
if IMAGE_TO_TEST in content_images:
    original_image = content_images[IMAGE_TO_TEST]
    num_models = len(models)

    # Calculate grid size (2 columns)
    num_rows = (num_models + 1) // 2 + 1 # +1 for original, +1 for rounding up

    plt.figure(figsize=(15, num_rows * 7.5))
    plt.suptitle(f"Style Comparison on '{IMAGE_TO_TEST}'", fontsize=20, y=1.02)

    # Plot Original
    ax = plt.subplot(num_rows, 2, 1)
    ax.imshow(original_image)
    ax.set_title("Original Image")
    ax.axis("off")

    # Plot Styled Images
    for i, (style_name, model) in enumerate(models.items()):
        styled_image = stylize_image(model, os.path.join(INPUT_DIR, IMAGE_TO_TEST))

        ax = plt.subplot(num_rows, 2, i + 2) # i + 2 to offset the original image
        ax.imshow(styled_image)
        ax.set_title(f"Style: {style_name}")
        ax.axis("off")

    plt.tight_layout()
    plot_path = os.path.join(OUTPUT_IMAGE_DIR, f"style_comparison_grid.jpg")
    plt.savefig(plot_path, bbox_inches='tight')
    plt.show() # Display the plot in the notebook
    print(f"Style comparison grid saved to {plot_path}")
else:
    print(f"Skipping style comparison: '{IMAGE_TO_TEST}' not found in Input folder.")

print("\nAll visualizations complete.")

Output hidden; open in https://colab.research.google.com to view.